In [1]:
import lamindb as ln
import ipywidgets as widgets
from IPython.display import display, clear_output
from vitessce import VitessceConfig, ViewType as qt, AnnDataWrapper

# --- 1. DÉFINITION DE LA FONCTION DE LANCEMENT (AVEC HEATMAP) ---
def launch_vitessce_lamin(artifact):
    # On récupère le chemin absolu via LaminDB
    zarr_path = artifact.path.as_posix()
    sample_id = artifact.description.replace("Viz: ", "")
    
    vc = VitessceConfig(schema_version="1.0.15", name=f"Analyse {sample_id}")
    
    dataset = vc.add_dataset(name=sample_id).add_object(
        AnnDataWrapper(
            adata_path=zarr_path,
            convert_to_zarr=False,
            obs_feature_matrix_path="X",
            obs_locations_path="obsm/X_spatial", 
            obs_embedding_paths=["obsm/X_umap"],
            obs_embedding_names=["UMAP"],
            # Ajout des deux sets de labels (Clusters + Cell Type)
            obs_set_paths=["obs/leiden", "obs/Type"],
            obs_set_names=["Clusters", "Cell Type"]
        )
    )

    # Définition des Vues
    spatial = vc.add_view(qt.SPATIAL, dataset=dataset)
    umap = vc.add_view(qt.SCATTERPLOT, dataset=dataset, mapping="UMAP")
    cell_sets = vc.add_view(qt.OBS_SETS, dataset=dataset)
    gene_list = vc.add_view(qt.FEATURE_LIST, dataset=dataset)
    heatmap = vc.add_view(qt.HEATMAP, dataset=dataset)

    # Mise en page : Spatial et UMAP en haut, Heatmap et listes en bas
    vc.layout(
        (spatial | umap) / (cell_sets | gene_list | heatmap)
    )
    
    return vc.display(proxy=True, height=800)

# --- 2. LOGIQUE DE L'INTERFACE LAMINDB (REVERSE QUERY) ---
all_viz = list(ln.Artifact.filter(description__contains="Viz:").all())
all_ulabels = list(ln.ULabel.filter().all())

statuts_possibles = ['Hot', 'Cold']
regions = ['Core', 'Edge', 'Control']
patients_possibles = sorted([l.name for l in all_ulabels if l.name not in statuts_possibles and l.name not in regions])

status_dd = widgets.Dropdown(options=statuts_possibles, description='📈 Statut:')
patient_dd = widgets.Dropdown(description='👤 Patient:')
sample_dd = widgets.Dropdown(description='📍 Échantillon:')
launch_btn = widgets.Button(description="Lancer l'Analyse", button_style='success')
out = widgets.Output()

def get_labels_for_art(artifact):
    return [l.name for l in ln.ULabel.filter(artifacts__id=artifact.id).all()]

def update_patients(*args):
    sel_status = status_dd.value
    valid_p = set()
    for a in all_viz:
        names = get_labels_for_art(a)
        if sel_status in names:
            for n in names:
                if n in patients_possibles: valid_p.add(n)
    patient_dd.options = sorted(list(valid_p))

def update_samples(*args):
    sel_patient = patient_dd.value
    options = {a.description.replace("Viz: ", "").replace(".zarr", ""): a for a in all_viz if sel_patient in get_labels_for_art(a)}
    sample_dd.options = options

status_dd.observe(update_patients, 'value')
patient_dd.observe(update_samples, 'value')

def on_click(b):
    with out:
        clear_output(wait=True)
        artifact = sample_dd.value
        if artifact:
            print(f"🚀 Chargement de {artifact.description}...")
            display(launch_vitessce_lamin(artifact))

launch_btn.on_click(on_click)

# Initialisation
update_patients()
update_samples()

# Affichage de l'interface
print("📋 7HILLS DASHBOARD (LaminDB Edition)")
display(widgets.VBox([status_dd, patient_dd, sample_dd, launch_btn, out]))

→ connected lamindb: anonymous/7Hills_Project
📋 7HILLS DASHBOARD (LaminDB Edition)
